In [20]:
import pandas, sklearn, xgboost


### API Helpers

In [21]:
import time, numpy as np, pandas as pd, requests

BASE = "https://api.jolpi.ca/ergast/f1"

def get_json(url):
    
    for i in range(6):
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            return r.json()
        time.sleep(3)
    raise RuntimeError("Could not get " + url)


def convert_to_seconds(t):
    if not isinstance(t, str) or ":" not in t:
        return np.nan 
    minutes, seconds = t.split(":")
    return int(minutes) * 60 + float(seconds)

    

### Race Results

In [22]:
rows = []

for year in range(2014, 2027):
    offset = 0
    while True:
        data = get_json(f"{BASE}/{year}/results.json?limit=100&offet={offset}")["MRData"]
        for race in data["RaceTable"]["Races"]:
            for r in race["Results"]:
                rows.append({
                    "year": year,
                    "round": int(race["round"]),
                    "date": race["date"],
                    "track": race["Circuit"]["circuitId"],
                    "driver" : r["Driver"]["driverId"],
                    "team": r["Constructor"]["constructorId"],
                    "grid": int(r["grid"]),
                    "finish": int(r["position"]),
                    "points": float(r["points"]),
                    "status": r["status"]
                })
        offset += 100
        time.sleep(0.5)
        if offset >= int(data["total"]):
            break
            
df = pd.DataFrame(rows)
df.to_csv("f1_results.csv", index = False)
print(df.shape)
print(df.tail())

(6100, 10)
      year  round        date       track      driver      team  grid  finish  \
6095  2026      5  2026-05-24  villeneuve       gasly    alpine    14       8   
6096  2026      5  2026-05-24  villeneuve       sainz  williams    15       9   
6097  2026      5  2026-05-24  villeneuve     bearman      haas    16      10   
6098  2026      5  2026-05-24  villeneuve     piastri   mclaren     4      11   
6099  2026      5  2026-05-24  villeneuve  hulkenberg      audi    11      12   

      points  status  
6095     4.0  Lapped  
6096     2.0  Lapped  
6097     1.0  Lapped  
6098     0.0  Lapped  
6099     0.0  Lapped  


### Qualifying Results

In [23]:
rows = []

for year in range(2014, 2027):
    offset = 0
    while True:
        data = get_json(f"{BASE}/{year}/qualifying.json?limit=100&offset={offset}")["MRData"]
        for race in data["RaceTable"]["Races"]:
            for r in race["QualifyingResults"]:
                rows.append({
                    "year": year,
                    "round": int(race["round"]),
                    "driver": r["Driver"]["driverId"],
                    "q1": r.get("Q1", ""),
                    "q2": r.get("Q2", ""),
                    "q3": r.get("Q3", "")
                })
        offset += 100
        time.sleep(0.5)
        if offset >= int(data["total"]):
                break
                        
quali = pd.DataFrame(rows)
quali.to_csv("f1_quali.csv", index = False)
print(quali.shape)
print(quali.tail())
            


(5502, 6)
      year  round        driver        q1 q2 q3
5497  2026     19        alonso  1:18.908      
5498  2026     19  ryo_hirakawa  1:18.958      
5499  2026     19        stroll  1:19.072      
5500  2026     19         perez  1:19.105      
5501  2026     19        bottas  1:19.852      


### Joining Results

In [24]:
results = pd.read_csv("f1_results.csv")
quali = pd.read_csv("f1_quali.csv")

def add_quali_gap(q):
    q = q.copy()
    
    for col in ["q1", "q2", "q3"]:
        q[col] = q[col].map(convert_to_seconds)
    best = q[["q1", "q2", "q3"]].min(axis = 1)
    pole = best.groupby([q["year"], q["round"]]).transform("min")
    q["quali_gap"] = (best - pole)/ pole * 100
    return q[["year", "round", "driver", "quali_gap"]]

data = results.merge(add_quali_gap(quali), on=["year", "round", "driver"], how="left")
data["quali_gap"] = data["quali_gap"].fillna(10).clip(upper=10)
data.to_csv("f1_data.csv", index = False)
print(data.shape)
print(data["quali_gap"].describe().round(2))
print(data[["year","round","driver","grid","quali_gap"]].tail(6))

(6100, 11)
count    6100.00
mean        2.12
std         1.84
min         0.00
25%         0.92
50%         1.78
75%         2.74
max        10.00
Name: quali_gap, dtype: float64
      year  round      driver  grid  quali_gap
6094  2026      5      lawson    12   1.817355
6095  2026      5       gasly    14   2.216925
6096  2026      5       sainz    15   2.335418
6097  2026      5     bearman    16   2.532448
6098  2026      5     piastri     4   0.279699
6099  2026      5  hulkenberg    11   1.802199
